In [ ]:
!pip install chardet

In [ ]:
# 確認 CSV 檔的格式
import chardet
with open("train_data.csv", "rb") as f:
    result = chardet.detect(f.read(10000))  # 偵測前 10 KB
print(result)

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd

# 載入 CSV
df = pd.read_csv("train_data.csv")


df_pos = df[df["labels"] == 1].sample(n=7500, random_state=42)  # 正面樣本
df_neg = df[df["labels"] == 0].sample(n=7500, random_state=42)  # 負面樣本

# 合併並打亂順序
df_balanced = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)

# 取出訓練集 1000 條，測試集 400 條
train_df = pd.concat([
    df_balanced[df_balanced["labels"]==1].iloc[:6000],
    df_balanced[df_balanced["labels"]==0].iloc[:6000]
]).sample(frac=1, random_state=42).reset_index(drop=True)

test_df = pd.concat([
    df_balanced[df_balanced["labels"]==1].iloc[6000:7500],
    df_balanced[df_balanced["labels"]==0].iloc[6000:7500]
]).sample(frac=1, random_state=42).reset_index(drop=True)


# 2️⃣ 轉成 Hugging Face Dataset
# dataset = Dataset.from_pandas(df)
# dataset = dataset.train_test_split(test_size=0.125)  # 90% 訓練, 10% 測試
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# 3️⃣ 定義標籤
label2id = {"Positive": 1, "Negative": 0}
id2label = {v:k for k,v in label2id.items()}


In [ ]:
def show_distribution(df, name="Dataset"):
    dist = df["labels"].value_counts().sort_index()
    ratio = df["labels"].value_counts(normalize=True).sort_index() * 100

    result = pd.DataFrame({
        "labels": dist.index,
        "Number": dist.values,
        "Fraction": ratio.round(1).astype(str) + "%"
    })

    print(f"\n{name} label distribution:")
    print(result.to_string(index=False))

# 使用範例
show_distribution(train_df, "Train")
show_distribution(test_df, "Test")


In [ ]:
# 5️⃣ 載入 tokenizer 與模型
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "IDEA-CCNL/Erlangshen-RoBERTa-110M-Sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# 6️⃣ Tokenize Text 欄位
def preprocess_function(examples):
    return tokenizer(examples["Text"], truncation=True, max_length=512)

#tokenized_dataset = dataset.map(preprocess_function, batched=True)
# 對訓練集 tokenization
tokenized_train = train_dataset.map(preprocess_function, batched=True)
# 對測試集 tokenization
tokenized_test = test_dataset.map(preprocess_function, batched=True)


# 2️⃣ 移除其他不必要欄位
# tokenized_dataset = tokenized_dataset.remove_columns(["Text", "Date", "Company", "Code", "Title", "Unnamed: 0"])
tokenized_dataset = tokenized_train.remove_columns(["Text", "Date", "Company", "Code", "Title", "Unnamed: 0"])
tokenized_dataset = tokenized_test.remove_columns(["Text", "Date", "Company", "Code", "Title", "Unnamed: 0"])
# 3️⃣ 設定 torch 格式，保留 labels
# tokenized_dataset.set_format(type="torch")
tokenized_train.set_format(type="torch")
tokenized_test.set_format(type="torch")


In [ ]:
# 7️⃣ 設定評估指標
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# 8️⃣ 設定訓練參數
from transformers import TrainingArguments, Trainer
# from transformers.integrations import EarlyStoppingCallback
training_args = TrainingArguments(
    output_dir="./my_bert_sentiment",
    eval_strategy="epoch",
    learning_rate=1.5e-5, # 降低學習率
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    optim='adamw_torch', # Specify PyTorch's AdamW optimizer for TPU compatibility
    report_to="none",       # ⚡ 不啟用 wandb
    load_best_model_at_end=True, # Load the best model at the end of training
    greater_is_better=False # For eval_loss, smaller is better
    #metric_for_early_stopping="eval_loss", # Metric to monitor for early stopping
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# 9️⃣ 建立 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # 若連續 3 次評估沒改善就停止
)

In [ ]:
trainer.train()

In [ ]:
import matplotlib.pyplot as plt
import json

# Load the trainer state which contains the training logs
trainer_state_path = "./my_bert_sentiment/checkpoint-2250/trainer_state.json"
with open(trainer_state_path, 'r') as f:
    trainer_state = json.load(f)
# Extract logging history
log_history = trainer_state['log_history']

# Extract steps and loss from the logging history
steps = [log['step'] for log in log_history if 'loss' in log]
losses = [log['loss'] for log in log_history if 'loss' in log]

# Plot the training loss
plt.figure(figsize=(10, 6))
plt.plot(steps, losses, label='Training Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Training Loss over Steps')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 評估
eval_results = trainer.evaluate()
print("Evaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

# 預測
preds_output = trainer.predict(tokenized_test)
preds = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids

In [ ]:
# 混淆矩陣
cm = confusion_matrix(labels, preds)

# Normalize the confusion matrix by row (to show percentage of each actual class)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Create the ConfusionMatrixDisplay
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=id2label)

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(cmap="Blues", ax=ax, values_format="d")

# 調整軸標籤字體大小
ax.tick_params(axis='x', labelsize=16)
ax.tick_params(axis='y', labelsize=16)

# 在格子內顯示百分比，文字顏色由百分比決定
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        percentage = cm_normalized[i, j] * 100

        # 百分比超過50%用白字，否則用黑字
        text_color = "white" if percentage > 50 else "black"

        # 顯示百分比
        ax.text(j, i + 0.15, f'({percentage:.2f}%)', ha="center", va="center",
                color=text_color, fontsize=16, weight='bold')


plt.title('Confusion Matrix', fontsize=16)
plt.show()


In [ ]:
# Make predictions on the train and test datasets
train_predictions = trainer.predict(tokenized_train)
test_predictions = trainer.predict(tokenized_test)

# Get the predicted labels (the index of the highest probability)
train_preds = np.argmax(train_predictions.predictions, axis=1)
test_preds = np.argmax(test_predictions.predictions, axis=1)

# Add the predictions as a new column to the original dataframes
train_df_with_preds = train_df.copy()
train_df_with_preds['predicted_labels'] = train_preds

test_df_with_preds = test_df.copy()
test_df_with_preds['predicted_labels'] = test_preds

# Save the dataframes with predictions to CSV files
train_df_with_preds.to_csv("train_data_with_predictions.csv", index=False)
test_df_with_preds.to_csv("test_data_with_predictions.csv", index=False)

print("Predictions saved to train_data_with_predictions.csv and test_data_with_predictions.csv")

In [ ]:
# 訓練結束後，定義存檔路徑
import shutil

save_path = "./fine-tuning"
# 1️⃣ 存儲模型與 tokenizer
trainer.save_model(save_path)                  # 保存模型權重和 config.json
tokenizer.save_pretrained(save_path)          # 保存 tokenizer 設定
# 2️⃣ 壓縮整個資料夾成 zip
shutil.make_archive("fine-tuned-model", 'zip', save_path)
print(f"模型資料夾已壓縮成 fine-tuned-model.zip")

# 壓縮 my_bert_sentiment 資料夾
shutil.make_archive("my_bert_sentiment", 'zip', "./my_bert_sentiment")
print("my_bert_sentiment 已壓縮成 my_bert_sentiment.zip")

In [ ]:
# 解壓縮後，可以直接用 from_pretrained 重新載入：
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassification.from_pretrained("./fine-tuning")
tokenizer = AutoTokenizer.from_pretrained("./fine-tuning")